# Day 3: Kafka Practice Exercises

## Welcome!
You've learned about Kafka architecture and basics—now let's practice! These exercises are designed for intermediate learners to build on basic Kafka operations. Try them on your own before checking the solutions. Some exercises use real-time data from the Wikipedia recent changes stream [https://stream.wikimedia.org/v2/stream/recentchange](https://stream.wikimedia.org/v2/stream/recentchange).

## Before You Start
- Ensure your Kafka container is running. Execute `docker-compose up zookeeper kafka` in the project folder using the updated `docker-compose.yml` (with Kafka 7.5.3 and Zookeeper 7.5.3).
- Open Jupyter Notebook at [http://localhost:8888](http://localhost:8888).
- Install the Kafka Python library if not installed: `!pip install kafka-python`.
- Install the `requests` library for the Wikimedia stream: `!pip install requests`.
- Use the bootstrap server `host.docker.internal:9093` as configured in the `docker-compose.yml`.

## Exercises

---


In [1]:
%pip install kafka-python requests

Note: you may need to restart the kernel to use updated packages.



### Exercise 1: Create a Topic
**What to Do**:
- Create a Kafka topic named `test-topic` with 1 partition and replication factor 1.

**Steps**:
1. Import `KafkaAdminClient` and `NewTopic` from `kafka.admin`.
2. Create an admin client with `bootstrap_servers="host.docker.internal:9093"`.
3. In a try-except block for `KafkaError`, create a topic list with `NewTopic(name="test-topic", num_partitions=1, replication_factor=1)` and use `admin_client.create_topics(new_topics=topic_list)`.
4. Verify by printing `admin_client.list_topics()`.

---


### Docs [KafkaAdminClient](https://kafka-python.readthedocs.io/en/master/apidoc/KafkaAdminClient.html)

In [1]:
from kafka.admin import KafkaAdminClient, NewTopic

In [2]:
admin_client = KafkaAdminClient(
    bootstrap_servers="host.docker.internal:9093",
)

In [3]:
from kafka.errors import KafkaError

In [4]:
# clean test-topics
admin_client.delete_topics(t for t in admin_client.list_topics() if t.startswith("test"))

DeleteTopicsResponse_v3(throttle_time_ms=0, topic_error_codes=[(topic='test-topic', error_code=0), (topic='test-topic-delayed', error_code=0), (topic='test-topic-2', error_code=0)])

In [5]:
try:
    topic_list = [
        NewTopic(name="test-topic", num_partitions=1, replication_factor=1),
        NewTopic(name="test-topic-2", num_partitions=1, replication_factor=1),
    ]
    admin_client.create_topics(new_topics=topic_list)
except KafkaError as e:
    print("Error during new topic creation:")
    print(e)        

In [6]:
admin_client.list_topics()

['test-topic', 'test-topic-2', 'my-topics', '__consumer_offsets']


### Exercise 2: List Topics
**What to Do**:
- List all Kafka topics.

**Steps**:
1. Print `admin_client.list_topics()`.

---


In [7]:
admin_client.list_topics()

['test-topic', 'test-topic-2', 'my-topics', '__consumer_offsets']


### Exercise 3: Produce a Message
**What to Do**:
- Send a message "Hello Kafka!" to `test-topic`.

**Steps**:
1. Import `KafkaProducer` from `kafka`.
2. Create a producer with `bootstrap_servers="host.docker.internal:9093"`.
3. Send the message using `producer.send('test-topic', b'Hello Kafka!')` and flush.
4. Print "Message sent!".

---


### Docs: [KafkaProducer](https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html)

In [8]:
from kafka import KafkaProducer
p = KafkaProducer(
    bootstrap_servers="host.docker.internal:9093",
)

In [9]:
p.send("test-topic", b"first message")
p.flush()
print("message sent")

message sent



### Exercise 4: Produce Multiple Messages
**What to Do**:
- Send three messages to `test-topic`.

**Steps**:
1. Define a list of messages: `[b'Message 1', b'Message 2', b'Message 3']`.
2. For each message, send it using `producer.send('test-topic', message)`, print the sent message, and flush after the loop.
3. Print "Three messages sent!".

---


In [10]:
msg_list = [b'Message 1', b'Message 2', b'Message 3']
for msg in msg_list:
    _msg = p.send("test-topic", msg)
    print(msg, "->", _msg)
p.flush()
print("3 messages sent")

b'Message 1' -> <kafka.producer.future.FutureRecordMetadata object at 0x7f2430394290>
b'Message 2' -> <kafka.producer.future.FutureRecordMetadata object at 0x7f2430394810>
b'Message 3' -> <kafka.producer.future.FutureRecordMetadata object at 0x7f2430394890>
3 messages sent


In [11]:
dir(_msg)[-16:]

['_produce_success',
 'add_both',
 'add_callback',
 'add_errback',
 'args',
 'chain',
 'error_on_callbacks',
 'exception',
 'failed',
 'failure',
 'get',
 'is_done',
 'retriable',
 'succeeded',
 'success',
 'value']

In [12]:
_msg.value

RecordMetadata(topic='test-topic', partition=0, topic_partition=TopicPartition(topic='test-topic', partition=0), offset=3, timestamp=1779200842865, checksum=None, serialized_key_size=-1, serialized_value_size=9, serialized_header_size=-1)


### Exercise 5: Consume a Message
**What to Do**:
- Read one message from `test-topic`.

**Steps**:
1. Import `KafkaConsumer` from `kafka`.
2. Create a consumer for 'test-topic' with `bootstrap_servers="host.docker.internal:9093"`, `auto_offset_reset='earliest'`.
3. In a for loop over consumer, print `message.value.decode('utf-8')` and break after one.

---


### Docs: [KafkaConsumer](https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html)

In [13]:
from kafka import KafkaConsumer
cons = KafkaConsumer(
    bootstrap_servers="host.docker.internal:9093",
    auto_offset_reset='earliest',
    consumer_timeout_ms=3_000,
)

In [14]:
# subscribe to topic with dynamical partition assignment
cons.subscribe(["test-topic", "test-topic-2", ])  # INCOMPATIBLE with .assign() !!

In [15]:
cons.subscription()

{'test-topic', 'test-topic-2'}

In [16]:
#p.send("test-topic", b"more meesaggeas"); p.flush()

In [17]:
for msg in cons:
    print(msg.value.decode('utf-8'))
    break
else:
    print("timeout! --> no more message.")

send some message


### Effect of flush in delayed message queue

In [24]:
p_delayed = KafkaProducer(
    bootstrap_servers="host.docker.internal:9093",
    linger_ms=5_000, # artificially create a delay larger than consumer timeout (3s)
)
consumer_delay_test = KafkaConsumer(
    "test-topic-delayed",
    bootstrap_servers="host.docker.internal:9093",
    auto_offset_reset='earliest',
    consumer_timeout_ms=3_000,
)

In [25]:
### WITHOUT FLUSH

# send a message from delayed producer
p_delayed.send("test-topic-delayed", b"delayed message")

# try to consume it: will fail because timeout of 3s < linger delay of 5s
for msg in consumer_delay_test:
    print(msg.value)
    break
else:
    print("timeout! --> no more message.")
    
# retry try to consume it: should work now because 2*3s > 5s
for msg in consumer_delay_test:
    print(msg.value)
    break
else:
    print("timeout! --> no more message.")


timeout! --> no more message.
b'delayed message'


In [26]:
### WITH FLUSH

# send a message from delayed producer and flush immediately
p_delayed.send("test-topic-delayed", b"flushed message")
p_delayed.flush()

# message will be available despite timeout of 3s < linger delay of 5s
for msg in consumer_delay_test:
    print(msg.value)
    break
else:
    print("timeout! --> no more message.")

b'flushed message'



### Exercise 6: Consume Multiple Messages
**What to Do**:
- Read three messages from `test-topic`.

**Steps**:
1. Initialize message_count = 0.
2. In a for loop over consumer, print the message, increment count, and break if count == 3.

---


In [18]:
num_msg_to_read = 3

for i in range(num_msg_to_read):
    for msg in cons:
        print(msg.value)
        break
    else:
        print("timeout! --> no more message.")

b'Message 1'
b'Message 2'
b'Message 3'



### Exercise 7: Create a Partitioned Topic
**What to Do**:
- Create a topic named `partitioned-topic` with 3 partitions.

**Steps**:
1. In a try-except for `KafkaError`, create topic_list with `NewTopic(name="partitioned-topic", num_partitions=3, replication_factor=1)` and use `admin_client.create_topics`.
2. Verify by printing `admin_client.list_topics()`.

---


In [19]:
try:
    topic_list = [
        NewTopic(name="partitioned-topic", num_partitions=3, replication_factor=1),
    ]
    admin_client.create_topics(topic_list)
except KafkaError as e:
    print("ERROR")
    
admin_client.list_topics()

['test-topic',
 'my-topics',
 '__consumer_offsets',
 'partitioned-topic',
 'test-topic-2']


### Exercise 8: Produce to Specific Partition
**What to Do**:
- Send a message to partition 1 of `partitioned-topic`.

**Steps**:
1. Send a message using `producer.send('partitioned-topic', b'Message to partition 1', partition=1)` and flush.
2. Print "Message sent to partition 1!".

--- 


In [64]:
result1 = p.send("partitioned-topic", b"Message-0 to partition 1", partition=1)
p.send("partitioned-topic", b"Message-A to partition 1", partition=1)
p.send("partitioned-topic", b"Message-B to partition 1", partition=1)
p.flush()
print("Message sent to partition 1!")

Message sent to partition 1!


In [65]:
p.send("partitioned-topic", b"Message to partition 2", partition=2)
p.send("partitioned-topic", b"Message1 to partition 2", partition=2)
p.send("partitioned-topic", b"Message2 to partition 2", partition=2)

p.send("partitioned-topic", b"Message to partition 0", partition=0)
p.send("partitioned-topic", b"Message1 to partition 0", partition=0)

p.flush()
print("Messages sent to partitions 0 and 2!")

Messages sent to partitions 0 and 2!



### Exercise 9: Consume from Specific Partition
**What to Do**:
- Read from partition 1 of `partitioned-topic`.

**Steps**:
1. Import `TopicPartition` from `kafka`.
2. Create consumer with `bootstrap_servers`, `auto_offset_reset='earliest'`.
3. Assign `[TopicPartition('partitioned-topic', 1)]` to consumer.
4. In a for loop, print the message and break after one.

--- 


In [25]:
from kafka import TopicPartition

In [29]:
consumer_partition_1 = KafkaConsumer(
    # "partitioned-topic",  # <- this assigns all partitions ?
    bootstrap_servers="host.docker.internal:9093",
    auto_offset_reset='earliest',
    consumer_timeout_ms=3_000,
)

In [41]:
part1 = TopicPartition(topic="partitioned-topic", partition=1)
part1, type(part1)

(TopicPartition(topic='partitioned-topic', partition=1),
 kafka.structs.TopicPartition)

In [42]:
# subscribe to specific topic partition manually
consumer_partition_1.assign([part1])  # INCOMPATIBLE with .subscribe() !!

In [47]:
consumer_partition_1.assignment()

{TopicPartition(topic='partitioned-topic', partition=1)}

In [46]:
consumer_partition_1.subscription()

In [66]:
for msg in consumer_partition_1:
    print(f"Topic: {msg.topic}, Partition: {msg.partition}, Value: {msg.value.decode('utf-8')}")
    #break
else:
    print("timeout! --> no more message.")

Topic: partitioned-topic, Partition: 1, Value: Message-0 to partition 1
Topic: partitioned-topic, Partition: 1, Value: Message-A to partition 1
Topic: partitioned-topic, Partition: 1, Value: Message-B to partition 1
timeout! --> no more message.


In [62]:
consumer_partition_0 = KafkaConsumer(
    bootstrap_servers="host.docker.internal:9093",
    auto_offset_reset='earliest',
    consumer_timeout_ms=3_000,
)
consumer_partition_0.assign([TopicPartition(topic="partitioned-topic", partition=0)])

consumer_partition_2 = KafkaConsumer(
    bootstrap_servers="host.docker.internal:9093",
    auto_offset_reset='earliest',
    consumer_timeout_ms=3_000,
)
consumer_partition_2.assign([TopicPartition(topic="partitioned-topic", partition=2)])

In [67]:
print("=== Messages on partition 0:")
for msg in consumer_partition_0:
    print(f"Topic: {msg.topic}, Partition: {msg.partition}, Value: {msg.value.decode('utf-8')}")
    #break
else:
    print("timeout! --> no more message.")
print()
print("=== Messages on partition 2:")
for msg in consumer_partition_2:
    print(f"Topic: {msg.topic}, Partition: {msg.partition}, Value: {msg.value.decode('utf-8')}")
    #break
else:
    print("timeout! --> no more message.")

=== Messages on partition 0:
Topic: partitioned-topic, Partition: 0, Value: Message to partition 0
Topic: partitioned-topic, Partition: 0, Value: Message1 to partition 0
timeout! --> no more message.

=== Messages on partition 2:
Topic: partitioned-topic, Partition: 2, Value: Message to partition 2
Topic: partitioned-topic, Partition: 2, Value: Message1 to partition 2
Topic: partitioned-topic, Partition: 2, Value: Message2 to partition 2
timeout! --> no more message.



### Exercise 10: Delete a Topic
**What to Do**:
- Delete the `test-topic`.

**Steps**:
1. In a try-except for `KafkaError`, use `admin_client.delete_topics(['test-topic'])`.
2. Verify by printing `admin_client.list_topics()`.

--- 


In [69]:
admin_client.list_topics()

['test-topic',
 'my-topics',
 '__consumer_offsets',
 'partitioned-topic',
 'test-topic-2']

In [74]:
try:
    admin_client.delete_topics(['test-topic', 'test-topic-2'])
except KafkaError as e:
    print("Error during deletion of at least one topic")
    print(e)

Error during deletion of at least one topic
[Error 3] UnknownTopicOrPartitionError: Request 'DeleteTopicsRequest_v3(topics=['test-topic', 'test-topic-2'], timeout=30000)' failed with response 'DeleteTopicsResponse_v3(throttle_time_ms=0, topic_error_codes=[(topic='test-topic', error_code=0), (topic='test-topic-2', error_code=3)])'.


In [75]:
admin_client.list_topics()

['partitioned-topic', 'my-topics', '__consumer_offsets']


### Exercise 11: Connect to Wikimedia Stream
**What to Do**:
- Connect to the Wikipedia recent changes stream and print the first message using a custom SSE client.

**Steps**:
1. Import `requests` and `json`.
2. Set url = 'https://stream.wikimedia.org/v2/stream/recentchange', headers with User-Agent and Accept.
3. In a with requests.get(stream=True), iterate over resp.iter_lines(decode_unicode=True).
4. If line.startswith('data: '), load json from line[6:], skip if 'meta' 'domain' == 'canary', print the event, and break after one.
5. Handle JSONDecodeError.


Notes:
* SSE = Server-Sent Events
* `"data"` in the response is the `event`

--- 


In [1]:
import  json
import requests
import datetime

In [2]:
url = 'https://stream.wikimedia.org/v2/stream/recentchange'
headers = {'User-Agent': 'WikiStreamBot/1.0', 'Accept': 'text/event-stream'}

In [60]:
!python --version
!type python

Python 3.11.6
python is /opt/conda/bin/python


In [73]:
from pprint import pprint

print_keys = set(("title", "wiki", 'server_url'))

def pretty_print_event_info(data: dict, num_event: int | None, *args, **kwargs) -> None:
    
    e = f" #{num_event:d}" if num_event is not None else ""
    
    print(f"=== Event{e}: ===")
    print_data = {"meta/domain": data.get("meta", {}).get("domain")}
    print_data.update({k: data[k] for k in data.keys() if k in print_keys})

    timestamp = data.get("timestamp")
    pretty_time = str(datetime.datetime.fromtimestamp(timestamp)) if timestamp else ""
    print_data.update({'datetime': pretty_time})
    
    pprint(print_data)
    print()

    return None

def main(
    event_handler: callable = pretty_print_event_info,
    max_events: int | None = None,
    debug_line_limit: int | None = None,
) -> None:
    """Custom SSE client"""
    
    if max_events is None:
        max_events = 5
        
    try:
        max_events = int(max_events)
    except:
        raise Exception("max_events must be convertable to integer")
    
    num_event = 0
    with requests.get(url, headers=headers, stream=True) as resp:
        for i, resp_line in enumerate(resp.iter_lines(decode_unicode=True)):
            
            # debug
            #print(f"=== response line {i:05d} ===")
            #print(resp_line)
    
            if not resp_line.startswith("data: "):
                continue
                
            try:
                data = json.loads(resp_line[6:])
            except json.JSONDecodeError as e:
                print("Error on decoding", repr(resp_line))
                print(e)
                print("skipping event...")
                continue
            
            if data.get("meta", {}).get("domain") == "canary":
                print(60*"=")
                print("we skip CANARY events!!! why though ?")
                pprint(data)
                print(60*"=")
                continue

            num_event += 1
    
            result = event_handler(data, num_event)

            if (
                (num_event >= max_events)
                or ((debug_line_limit is not None) and (i > int(debug_line_limit)))
            ):
                break
                
main(max_events=2)


=== Event #1: ===
{'datetime': '2026-05-19 18:17:30',
 'meta/domain': 'commons.wikimedia.org',
 'server_url': 'https://commons.wikimedia.org',
 'title': 'User:Edelseider',
 'wiki': 'commonswiki'}

=== Event #2: ===
{'datetime': '2026-05-19 18:17:30',
 'meta/domain': 'www.wikidata.org',
 'server_url': 'https://www.wikidata.org',
 'title': 'Q23038394',
 'wiki': 'wikidatawiki'}




### Exercise 12: Extract User from Wikimedia Stream
**What to Do**:
- Connect to the Wikipedia recent changes stream and print the "user" field from the first 3 events using a custom SSE client.

**Steps**:
1. Similar to Exercise 11, but count events, print event['user'], and break after 3 events.

--- 


In [71]:
def extract_user_info(data: dict, num_event: int | None, *args, **kwargs):
    print(f"""User{num_event:03d}: {data.get("user")}""")
          
main(event_handler=extract_user_info, max_events=13)

User001: Morkoz
User002: DPLA bot
User003: SuperGrey-bot
User004: Elanyvx
User005: SuperGrey-bot
User006: SuperGrey-bot
User007: DOPBot
User008: SuperGrey-bot
User009: Balyozbot
User010: SuperGrey-bot
User011: DOPBot
User012: SuperGrey-bot
User013: Mathieu Kappler



### Exercise 13: Count Edits in Wikimedia Stream
**What to Do**:
- Connect to the Wikipedia recent changes stream and count the first 10 events of type "edit" using a custom SSE client.

**Steps**:
1. Similar setup, initialize edit_count = 0, total_events = 0.
2. For each event, increment total_events, `if event.get('type') == 'edit'`, increment edit_count.
3. After 10 total_events, print edit_count and break.

--- 


In [83]:
class EditsCounter:
    NUM_EDITS = 0

    @property
    def num_edits(self):
        return self.NUM_EDITS
    
    def __call__(self, data: dict, num_event: int | None, *args, **kwargs):
        print(data.get("type"))
        #pprint(data)
        #print()
        if data.get("type") == "edit":
            self.NUM_EDITS += 1
        return None

edits_counter = EditsCounter()
num_events = 10
main(edits_counter, num_events)

print("\nDone counting!\n")
print(f"We got {edits_counter.num_edits} edits in {num_events} events")

edit
edit
log
categorize
categorize
edit
categorize
edit
categorize
categorize

Done counting!

We got 4 edits in 10 events



### Exercise 14: Configure Topic with Custom Retention and Segment Size
**What to Do**:
- Create a topic named retention-topic with a retention period of 2 hours and a segment size of 10MB, and verify the topic creation.

**Steps**:
1. Import KafkaAdminClient, NewTopic, and TopicAlreadyExistsError from kafka.admin and kafka.errors.
2. Create an admin client with bootstrap_servers="host.docker.internal:9093", client_id='retention_topic_creator'.
3. In a try-except block for TopicAlreadyExistsError, create a topic list with NewTopic(name="retention-topic", num_partitions=1, replication_factor=1, topic_configs={'retention.ms': '7200000', 'segment.bytes': '10485760', 'cleanup.policy': 'delete'}) and use admin_client.create_topics(new_topics=topic_list, validate_only=False).
4. Print confirmation of topic creation or handle if the topic already exists.
5. Verify by printing admin_client.list_topics().

--- 


In [84]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

In [85]:
admin_client = KafkaAdminClient(
    bootstrap_servers="host.docker.internal:9093",
    client_id="retention_topic_creator",
)

In [119]:
try:
    res = admin_client.delete_topics(['retention-topic'])
    print(res)
except:
    print("nothing done")

DeleteTopicsResponse_v3(throttle_time_ms=0, topic_error_codes=[(topic='retention-topic', error_code=0)])


In [120]:
try:
    topic_list = [
        NewTopic(
            name="retention-topic",
            num_partitions=1,
            replication_factor=1,
            topic_configs={
                'retention.ms': '7200000',
                'segment.bytes': '10485760',
                'cleanup.policy': 'delete',
            }),
    ]
    admin_client.create_topics(new_topics=topic_list, validate_only=False)
    print("Topic created successfully")
except TopicAlreadyExistsError as e:
    print("Error during new topic creation:")
    print(e)      

Topic created successfully


In [112]:
admin_client.list_topics()

['partitioned-topic', 'my-topics', 'retention-topic', '__consumer_offsets']


### Exercise 15: Produce JSON Messages with Custom Serializer
**What to Do**:
- Send 3 JSON messages to `retention-topic`, each containing fields `event_id`, `category`, and `timestamp`, using a custom JSON serializer.

**Steps**:
1. Create producer with value_serializer=lambda x: json.dumps(x).encode('utf-8').
2. Define 3 messages like {'event_id': 1, 'category': 'sale', 'timestamp': '2025-09-18T10:00:00'}.
3. Send each and flush, print "Three JSON messages sent!".

--- 


In [113]:
from kafka import KafkaProducer, KafkaConsumer

In [114]:
retention_producer = KafkaProducer(
    bootstrap_servers="host.docker.internal:9093",
    value_serializer=lambda x: json.dumps(x).encode('utf-8'),
)

In [135]:
msgs = [
    {'event_id': 1, 'category': 'sale', 'timestamp': '2025-09-18T09:00:00'},
    {'event_id': 2, 'category': 'discount', 'timestamp': '2025-09-18T10:00:00'},
    {'event_id': 3, 'category': 'sale', 'timestamp': '2025-09-18T10:11:00'},
    {'event_id': 4,                     'timestamp': '2025-09-18T10:16:00'},
    {'event_id': 5, 'category': 'sale', 'timestamp': '2025-09-18T10:20:00'},
]

In [141]:
for msg in msgs:
    retention_producer.send("retention-topic", msg)
retention_producer.flush()
print("Three JSON messages sent!")

Three JSON messages sent!



### Exercise 16: Consume and Validate JSON Messages
**What to Do**:
- Read 3 JSON messages from `retention-topic`, validate that they contain required fields, and print the `category` field.

**Steps**:
1. Create consumer with value_deserializer=lambda x: json.loads(x.decode('utf-8')), auto_offset_reset='earliest'.
2. In loop, for 3 messages, check if 'event_id', 'category', 'timestamp' in message.value, print category or "Invalid message".

--- 


In [122]:
retention_consumer = KafkaConsumer(
    "retention-topic",
    bootstrap_servers="host.docker.internal:9093",
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    auto_offset_reset='earliest',
    consumer_timeout_ms=3_000,
)

In [142]:
required_fields = ['event_id', 'category', 'timestamp']

def message_is_valid(msg):
    for f in required_fields:
        if not f in msg.value:
            return False
    return True

for i in range(5):
    for msg in retention_consumer:
        if not message_is_valid(msg):
            print("Invalid message")
            continue
        print(f"""Category: {msg.value["category"]}""")
        break

Category: sale
Category: discount
Category: sale
Invalid message
Category: sale



### Exercise 17: Distribute Messages Across Consumer Group
**What to Do**:
- Simulate two consumers in the same consumer group reading from `partitioned-topic` (3 partitions) and print the partition and message for each.

**Steps**:
1. First, produce messages to partitions if needed (as in solution).
2. Create two consumers with same group_id='my-group', auto_offset_reset='earliest'.
3. For each consumer, in loop print partition and message, break after one or as needed. Note: Simulate in separate cells.

--- 


In [28]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError
from kafka import KafkaProducer
import time

# clean up topic
admin_client = KafkaAdminClient(
    bootstrap_servers="host.docker.internal:9093",
    client_id="partition-topic-creator",
)


try:
    res = admin_client.delete_topics(['partition-topic'])
    print(res)
except:
    print("Topic does not exist - nothing done")

print("Topics after cleanup:", admin_client.list_topics())

time.sleep(1)
try:
    topic_list = [
        NewTopic(
            name="partition-topic",
            num_partitions=3,
            replication_factor=1,
        )]

    admin_client.create_topics(new_topics=topic_list, validate_only=False)
    print("Topic created successfully")
    
except TopicAlreadyExistsError as e:
    print("Error during new topic creation:")
    print(e)

print("Topics:", admin_client.list_topics())

# create basic producer
producer = KafkaProducer(
    bootstrap_servers="host.docker.internal:9093"
)


DeleteTopicsResponse_v3(throttle_time_ms=0, topic_error_codes=[(topic='partition-topic', error_code=0)])
Topics after cleanup: ['partitioned-topic', 'my-topics', 'retention-topic', '__consumer_offsets']
Topic created successfully
Topics: ['my-topics', 'retention-topic', '__consumer_offsets', 'partitioned-topic', 'partition-topic']


In [29]:
# send messages
producer.send("partitioned-topic", b"Message0 to partition 0", partition=0)
producer.send("partitioned-topic", b"Message1 to partition 0", partition=0)

producer.send("partitioned-topic", b"Message-0 to partition 1", partition=1)
producer.send("partitioned-topic", b"Message-A to partition 1", partition=1)
producer.send("partitioned-topic", b"Message-B to partition 1", partition=1)
producer.send("partitioned-topic", b"Message-C to partition 1", partition=1)
producer.send("partitioned-topic", b"Message-D to partition 1", partition=1)
producer.send("partitioned-topic", b"Message-E to partition 1", partition=1)

producer.send("partitioned-topic", b"Message-0 to partition 2", partition=2)
producer.send("partitioned-topic", b"Message-1 to partition 2", partition=2)
producer.send("partitioned-topic", b"Message-2 to partition 2", partition=2)

producer.flush()
print("Messages sent to partitions 0,1,2!")

Messages sent to partitions 0,1,2!


In [27]:
producer.send("partitioned-topic", b"Message-0 to partition 2", partition=2)
producer.send("partitioned-topic", b"Message-1 to partition 2", partition=2)
producer.send("partitioned-topic", b"Message-2 to partition 2", partition=2)

producer.flush()
print("Messages sent to partitions 2!")

Messages sent to partitions 2!


In [31]:

# Note: Run Consumer 1 in a separate cell or script
from kafka import KafkaConsumer
consumer1 = KafkaConsumer(
    'partitioned-topic',
    bootstrap_servers="host.docker.internal:9093",
    group_id='test-group',
    auto_offset_reset='earliest'
)
print("Consumer 1 started...")
for message in consumer1:
    print(f"Consumer 1, Partition {message.partition}: {message.value.decode('utf-8')}")
    break

# Note: Run Consumer 2 in a separate cell or script
from kafka import KafkaConsumer
consumer2 = KafkaConsumer(
    'partitioned-topic',
    bootstrap_servers="host.docker.internal:9093",
    group_id='test-group',
    auto_offset_reset='earliest'
)
print("Consumer 2 started...")
for message in consumer2:
    print(f"Consumer 2, Partition {message.partition}: {message.value.decode('utf-8')}")
    break

Consumer 1 started...
Consumer 1, Partition 0: Message0 to partition 0
Consumer 2 started...


KeyboardInterrupt: 


### Exercise 18: Filter Wikimedia Stream by Edit Size
**What to Do**:
- Connect to the Wikipedia stream and print the first 3 edit events where the edit size (`length['new'] - length['old']`) is greater than 100 bytes.

**Steps**:
1. Similar stream setup.
2. Initialize large_edit_count = 0.
3. For each event, if type == 'edit', calculate edit_size = event['length']['new'] - event['length']['old'], if >100, print title, user, size, increment count, break if ==3.

--- 



### Exercise 19: Produce Key-Based Messages with Consistent Partitioning
**What to Do**:
- Send 5 messages to partitioned-topic with keys alice, bob, and charlie to ensure messages with the same key go to the same partition, and print the partition each message is sent to.

**Steps**:

1. Import KafkaProducer from kafka and json.
2. Create a producer with bootstrap_servers='host.docker.internal:9093', key_serializer=lambda x: json.dumps(x).encode('utf-8'), value_serializer=lambda x: json.dumps(x).encode('utf-8').
3. Define a list of tuples: `[('alice', 'Message 1 from alice'), ('bob', 'Message 1 from bob'), ('alice', 'Message 2 from alice'), ('charlie', 'Message 1 from charlie'), ('bob', 'Message 2 from bob')]`.
4. For each tuple, send the message using `producer.send('partitioned-topic', key=key, value=value)`, print the key and partition (from future.get().partition), and flush after the loop.
5. Print "Five key-based messages sent!".

---



### Exercise 20: Consume with Manual Offset Commit
**What to Do**:
- Create a consumer for partitioned-topic that manually commits offsets after processing each of 2 messages.

**Steps**:

1. Import KafkaConsumer from kafka.
2. Create a consumer with `bootstrap_servers='host.docker.internal:9093', group_id='my-consumer-group', enable_auto_commit=False, auto_offset_reset='earliest',           value_deserializer=lambda x: x.decode('utf-8')`.
3. In a loop, for 2 messages, print the received message, commit with consumer.commit(), print the committed partition, and break after 2 messages.
4. Close the consumer.

--- 



### Exercise 21: Stream and Aggregate Wikimedia Events to Kafka
**What to Do**:
- Connect to the Wikipedia stream, aggregate edit counts by `wiki` for the first 10 events, and produce the aggregated data to a new topic `wiki-aggregates`.

**Steps**:
1. Create 'wiki-aggregates' topic with 1 partition.
2. Create producer with json serializer.
3. Stream setup, initialize edit_counts={}, total_events=0.
4. For each event, skip canary, increment total, if type=='edit', increment edit_counts[wiki].
5. When total==10, send {'edit_counts': edit_counts, 'total_events': total_events} to topic, print sent, break.
6. Flush and print complete.

--- 



### Exercise 22: Consume Aggregate Wikimedia Events from `wiki-aggregates` topic.
**What to Do**:
- Create a Kafka consumer to read and display the aggregated Wikipedia edit data from the wiki-aggregates topic that was previously produced by the streaming application.

**Steps**:
1. Import KafkaConsumer, TopicPartition, json.
2. Create consumer with bootstrap, consumer_timeout_ms=10000, value_deserializer json.
3. Assign TopicPartition('wiki-aggregates', 0), seek_to_beginning.
4. In try, loop over consumer, print message details: edit_counts, total_events, partition, offset.
5. Print total messages read, handle exceptions, finally close consumer.



## Finish Up
- Save your notebook as `exercises/03_exercises.ipynb`.
- Stop containers with `docker compose down` if needed.

## Tips
- Verify Kafka is running with `docker ps`.
- Use `print()` to monitor results after each step.
- Save frequently in Jupyter.
- The Wikimedia stream requires an internet connection and the `requests` library.